In [ ]:
# Install packages (skip if the environment already has them)
%pip -q install --no-cache-dir "vllm>=0.22" "datasets>=3.0" "aiohttp>=3.9" "math-verify[antlr4_13_2]" "pandas<3" scikit-learn transformers


In [ ]:
# Config: every experiment decision, fixed before any run and saved to prereg.json
from pathlib import Path
CFG = dict(
    model='Qwen/Qwen3-1.7B', dtype='bfloat16', gpu='RTX A6000',
    work_dir=str(Path.cwd() / 'latte_e2'),
    labels_dir='labels',
    budgets=[0, 256, 512, 1024, 2048],
    nothink_max_tokens=1024,
    actions=[[1, 0], [4, 0], [8, 0], [1, 1024], [1, 2048], [4, 2048]],
    split_seed=7, n_dev=100,
    slo_s=60.0, slo_safety=0.9,
    server=dict(port=8015, max_model_len=4096, max_num_seqs=64, gpu_mem=0.80, prefix_caching=True),
    ladder=dict(reference_action=[1, 1024], multipliers=[0.25, 0.5, 0.75, 1.0, 1.25, 1.5], window_s=300, burst=32),
    step_b=dict(n_requests=100),
    step_c=dict(near_band=0.30, tolerance=0.10),
    aa=dict(n_pairs=5, seed_start=100),
    e2=dict(seeds=[1, 2, 3], n_requests=200, warmup=20, conditions=['light', 'contended', 'ramp']),
    mmg=dict(practical_floor=0.10, z_total=2.8),
    router=dict(probe_live=True, rate_window_s=30, feedback_window=20, feedback_step=1.2),
    probe=dict(outer_folds=5, inner_folds=3, layer_step=4),
    timeout_s=900,
)


In [ ]:
# Setup: work folder, prereg check, vLLM server, request/grading/telemetry helpers, smoke test
import os, json, time, math, random, asyncio, subprocess, re
from collections import deque
from functools import lru_cache
import numpy as np, pandas as pd, requests, aiohttp

WD = Path(CFG['work_dir']); WD.mkdir(parents=True, exist_ok=True)
RUNS = WD / 'runs'; RUNS.mkdir(exist_ok=True)
os.environ.setdefault('HF_HOME', str(WD / 'hf_cache'))
PRE = WD / 'prereg.json'
if PRE.exists():
    assert json.load(open(PRE)) == CFG, 'CFG differs from prereg.json: use a new work_dir instead of editing a frozen setup'
else:
    json.dump(CFG, open(PRE, 'w'), indent=1)

ACTIONS = [tuple(a) for a in CFG['actions']]
aname = lambda a: f'{ACTIONS[a][0]}x{ACTIONS[a][1]}'
NA, SLO = len(ACTIONS), CFG['slo_s']
URL, SERVED = f"http://127.0.0.1:{CFG['server']['port']}", 'latte'
USER_TMPL = ('Solve the following math problem carefully. Think through the steps internally. '
             'After finishing your thinking, give ONLY the final answer formatted as '
             '\\boxed{ANSWER}, with no additional explanation in the final response.\n\n{problem}')
SAMPLING = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20}


def healthy():
    try:
        return requests.get(URL + '/health', timeout=3).status_code == 200
    except requests.RequestException:
        return False


if not healthy():
    s = CFG['server']
    cmd = ['vllm', 'serve', CFG['model'], '--host', '127.0.0.1', '--port', str(s['port']), '--served-model-name', SERVED,
           '--dtype', CFG['dtype'], '--reasoning-parser', 'qwen3', '--max-model-len', str(s['max_model_len']),
           '--max-num-seqs', str(s['max_num_seqs']), '--gpu-memory-utilization', str(s['gpu_mem']),
           '--reasoning-config', json.dumps({'reasoning_start_str': '<think>', 'reasoning_end_str':
               r'Thinking time is over. Output only \boxed{ANSWER} now, with no further reasoning.</think>'}),
           '--enable-prefix-caching' if s['prefix_caching'] else '--no-enable-prefix-caching']
    SERVER = subprocess.Popen(cmd, stdout=open(WD / 'vllm.log', 'a'), stderr=subprocess.STDOUT)
    t_end = time.time() + 900
    while not healthy():
        assert SERVER.poll() is None and time.time() < t_end, (WD / 'vllm.log').read_text()[-3000:]
        time.sleep(5)
import shutil
print('server ready | GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True,
                                             text=True).stdout.strip() if shutil.which('nvidia-smi') else 'nvidia-smi not found')

from transformers import AutoTokenizer
TOKZ = AutoTokenizer.from_pretrained(CFG['model'])


def chat(problem, thinking):
    return TOKZ.apply_chat_template([{'role': 'user', 'content': USER_TMPL.replace('{problem}', problem)}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=thinking)


def last_boxed(text):
    i = (text or '').rfind('\\boxed')
    j = text.find('{', i) if i >= 0 else -1
    if j < 0:
        return None
    d = 0
    for k in range(j, len(text)):
        d += (text[k] == '{') - (text[k] == '}')
        if d == 0:
            return text[j + 1:k]
    return None


def norm_key(s):
    if s is None:
        return None
    t = re.sub(r'\\text\{([^}]*)\}', r'\1', s.strip())
    for a in ['\\left', '\\right', '\\!', '\\,', '\\;', '\\ ', '$', '^\\circ', '^{\\circ}', '\\%', '%']:
        t = t.replace(a, '')
    t = re.sub(r'\s+', '', t.replace('dfrac', 'frac').replace('tfrac', 'frac')).rstrip('.')
    return t or None


@lru_cache(maxsize=200000)
def is_correct(gold, pred):
    try:
        from math_verify import parse, verify
        if verify(parse('\\boxed{%s}' % gold), parse('\\boxed{%s}' % pred)):
            return True
    except Exception:
        pass
    return norm_key(gold) == norm_key(pred)


def vote_correct(preds, gold, rng):
    counts, first = {}, {}
    for j, p in enumerate(preds):
        k = norm_key(p)
        if k is not None:
            counts[k] = counts.get(k, 0) + 1
            first.setdefault(k, j)
    if not counts:
        return False
    top = [k for k, c in counts.items() if c == max(counts.values())]
    return bool(is_correct(gold, preds[first[rng.choice(top)]]))


async def post(session, path, payload):
    try:
        async with session.post(URL + path, json=payload) as r:
            body = await r.text()
            return (True, json.loads(body)) if r.status == 200 else (False, f'HTTP {r.status}: {body[:300]}')
    except Exception as e:
        return False, repr(e)


def used(d):
    return int((d.get('usage') or {}).get('completion_tokens') or 0)


async def serve(session, problem, n, budget, seed):
    msgs = [{'role': 'user', 'content': USER_TMPL.replace('{problem}', problem)}]
    if budget == 0:
        ok, d = await post(session, '/v1/chat/completions', {'model': SERVED, 'messages': msgs, 'n': n, 'seed': seed,
                           'max_tokens': CFG['nothink_max_tokens'], 'chat_template_kwargs': {'enable_thinking': False}, **SAMPLING})
        if not ok:
            return dict(ok=False, preds=[], tokens=0, error=d)
        return dict(ok=True, preds=[last_boxed((c.get('message') or {}).get('content') or '') for c in d['choices']],
                    tokens=used(d), error=None)
    ok, d = await post(session, '/v1/chat/completions', {'model': SERVED, 'messages': msgs, 'n': n, 'seed': seed,
                       'max_tokens': budget + 32, 'thinking_token_budget': budget,
                       'chat_template_kwargs': {'enable_thinking': True}, **SAMPLING})
    if not ok:
        return dict(ok=False, preds=[], tokens=0, error='stage1 ' + d)

    async def final(i, c):
        reasoning = ((c.get('message') or {}).get('reasoning') or (c.get('message') or {}).get('reasoning_content') or '')
        reasoning = reasoning.replace('<think>', '').split('</think>')[0].strip()
        if not reasoning:
            return None, 0, 'no reasoning'
        prefix = chat(problem, True)
        prefix += '' if prefix.rstrip().endswith('<think>') else '<think>\n'
        ok2, f = await post(session, '/v1/completions', {'model': SERVED, 'prompt': prefix + reasoning + '\n</think>\n\n\\boxed{',
                            'max_tokens': 256, 'temperature': 0.0, 'seed': seed + i})
        if not ok2:
            return None, 0, f
        return last_boxed('\\boxed{' + f['choices'][0]['text']), used(f), None

    fin = await asyncio.gather(*(final(i, c) for i, c in enumerate(d['choices'])))
    errs = [e for _, _, e in fin if e]
    return dict(ok=not errs, preds=[p for p, _, _ in fin], error=errs[0] if errs else None,
                tokens=used(d) + sum(t for _, t, _ in fin))


def prom(txt, name):
    v = [float(l.split()[-1]) for l in txt.splitlines() if l.startswith(name) and l[len(name):len(name) + 1] in ' {']
    return sum(v) if v else None


async def telemetry(session, stop, rows):
    while not stop.is_set():
        row = {'t': time.perf_counter()}
        try:
            async with session.get(URL + '/metrics') as r:
                txt = await r.text()
            row.update(running=prom(txt, 'vllm:num_requests_running'), waiting=prom(txt, 'vllm:num_requests_waiting'),
                       kv=prom(txt, 'vllm:kv_cache_usage_perc'))
        except Exception:
            pass
        rows.append(row)
        try:
            await asyncio.wait_for(stop.wait(), 1.0)
        except asyncio.TimeoutError:
            pass


def session():
    return aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=CFG['timeout_s']),
                                 connector=aiohttp.TCPConnector(limit=1024))


async with session() as s_:
    smoke = await serve(s_, 'Compute 17 times 19.', 1, 256, 1)
assert smoke['ok'] and smoke['preds'][0] in ('323', '323.0'), smoke
print('smoke test ok')


In [ ]:
# Labels: per-question vote accuracy for each action, dev / held-out split
from datasets import load_dataset
LAB = Path(CFG['labels_dir']) if Path(CFG['labels_dir']).is_absolute() else WD / CFG['labels_dir']
recs = [r for f in sorted(LAB.glob('chunk_*.json')) for r in json.load(open(f))]
assert recs, f'no label chunks in {LAB}'
PROBLEM = {str(r['unique_id']): r['problem'] for r in load_dataset('HuggingFaceH4/MATH-500', split='test')}
Q = len(recs)
GOLD = [r['gold'] for r in recs]
TEXT = [PROBLEM[r['id']] for r in recs]
BUD = CFG['budgets']
K = len(recs[0]['traces'])
assert all(n <= K for n, _ in ACTIONS) and all(b in BUD for _, b in ACTIONS)

rng_l = np.random.default_rng(0)
PF = np.zeros((Q, NA))
for i, r in enumerate(recs):
    for a, (n, b) in enumerate(ACTIONS):
        tr = [t['b'][str(b)] for t in r['traces']]
        subs = [[k] for k in range(K)] if n == 1 else [list(range(K))] if n == K else \
            [rng_l.choice(K, n, replace=False) for _ in range(48)]
        hits = []
        for sub in subs:
            keys = [tr[k]['key'] for k in sub]
            cnt = {k: keys.count(k) for k in keys if k is not None}
            if not cnt:
                hits.append(False)
                continue
            top = max(cnt.values())
            hits.append(np.mean([tr[sub[keys.index(k)]]['ok'] for k, c in cnt.items() if c == top]))
        PF[i, a] = np.mean(hits)

order = np.random.default_rng(CFG['split_seed']).permutation(Q)
DEV, HELD = np.sort(order[:CFG['n_dev']]), np.sort(order[CFG['n_dev']:])
cap_hits = np.mean([t['b']['0']['ft'] >= CFG['nothink_max_tokens'] for r in recs for t in r['traces']]) if 0 in BUD else 0
print(f'{Q} questions ({len(DEV)} dev / {len(HELD)} held-out); B=0 samples at the token cap: {cap_hits:.1%}')
print(pd.DataFrame({'action': [aname(a) for a in range(NA)], 'label_acc_dev': PF[DEV].mean(0),
                    'label_acc_held': PF[HELD].mean(0)}).round(3).to_string(index=False))


In [ ]:
# Probe: prefill hidden states, nested cross-fitted predictions (PHAT), live prediction function
import torch
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

HFM = __import__('transformers').AutoModelForCausalLM.from_pretrained(
    CFG['model'], torch_dtype=getattr(torch, CFG['dtype'])).to('cuda').eval()
LAYERS = list(range(0, HFM.config.num_hidden_layers + 1, CFG['probe']['layer_step']))


def hidden(q):
    ids = TOKZ(chat(TEXT[q], True), return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        hs = HFM(**ids, output_hidden_states=True).hidden_states
    return torch.stack([hs[l][0, -1] for l in LAYERS]).float().cpu().numpy()


hid_path = WD / 'hidden.npz'
if hid_path.exists():
    HID = np.load(hid_path)['X']
else:
    HID = np.stack([hidden(q) for q in range(Q)])
    np.savez_compressed(hid_path, X=HID)
ms = []
for q in DEV[:30]:
    torch.cuda.synchronize(); t0 = time.perf_counter(); hidden(q); torch.cuda.synchronize()
    ms.append((time.perf_counter() - t0) * 1000)
PROBE_MS = float(np.median(ms))


def fit(X, Y):
    sc = StandardScaler().fit(X)
    return sc, RidgeCV(alphas=np.logspace(0, 5, 11)).fit(sc.transform(X), Y)


FOLD_MODEL = {}
PHAT = np.zeros_like(PF)
for tr, te in KFold(CFG['probe']['outer_folds'], shuffle=True, random_state=CFG['split_seed']).split(np.arange(Q)):
    err = []
    for li in range(len(LAYERS)):
        e = 0.0
        for itr, ite in KFold(CFG['probe']['inner_folds'], shuffle=True, random_state=1).split(tr):
            sc, m = fit(HID[tr[itr], li], PF[tr[itr]])
            e += ((m.predict(sc.transform(HID[tr[ite], li])) - PF[tr[ite]]) ** 2).mean()
        err.append(e)
    li = int(np.argmin(err))
    sc, m = fit(HID[tr, li], PF[tr])
    PHAT[te] = np.clip(m.predict(sc.transform(HID[te, li])), 0, 1)
    for q in te:
        FOLD_MODEL[int(q)] = (li, sc, m)

PROBE_LOCK = asyncio.Lock()


async def probe(q):
    if not CFG['router']['probe_live']:
        return PHAT[q]
    li, sc, m = FOLD_MODEL[q]
    async with PROBE_LOCK:
        h = await asyncio.to_thread(hidden, q)
    return np.clip(m.predict(sc.transform(h[li][None]))[0], 0, 1)


base = np.abs(PF - PF.mean(0)).mean()
print(f'probe layers chosen per fold: {sorted({LAYERS[v[0]] for v in FOLD_MODEL.values()})} | '
      f'mean abs error {np.abs(PHAT - PF).mean():.3f} vs {base:.3f} for the mean predictor | live cost {PROBE_MS:.0f} ms')


In [ ]:
# Shared runner: Poisson or ramp arrivals, one run of one policy, run summary
def arrivals(spec, n, seed):
    g, t, out = random.Random(seed), 0.0, []
    if spec[0] == 'const':
        while len(out) < n:
            t += g.expovariate(spec[1])
            out.append(t)
        return out
    r0, r1 = spec[1], spec[2]
    T = n / ((r0 + r1) / 2)
    while len(out) < n:
        t += g.expovariate(max(r0, r1))
        if g.random() < (r0 + (r1 - r0) * min(t / T, 1.0)) / max(r0, r1):
            out.append(t)
    return out


def questions(pool, n, seed):
    p = np.random.default_rng(seed).permutation(pool)
    return [int(p[i % len(p)]) for i in range(n)]


async def run(tag, policy, spec, seed, pool, n, rep=0):
    path = RUNS / f'{tag}__s{seed}__r{rep}.json'
    if path.exists():
        return json.load(open(path))
    qs, ts = questions(pool, n, seed), arrivals(spec, n, seed)
    st = dict(arrivals=deque(), lat=deque(maxlen=CFG['router']['feedback_window']), mult=1.0,
              rng=random.Random(seed), vote_rng=random.Random(seed + 1))
    tel, recs_ = [], []
    async with session() as s:
        stop = asyncio.Event()
        mon = asyncio.create_task(telemetry(s, stop, tel))
        await asyncio.sleep(1.5)
        t0 = st['t0'] = time.perf_counter()

        async def one(j, q):
            arrive = time.perf_counter()
            st['arrivals'].append(arrive)
            snap = tel[-1] if tel else {}
            a, lam = await policy['decide'](q, st, arrive)
            decided = time.perf_counter()
            n_, b_ = ACTIONS[a]
            out = await serve(s, TEXT[q], n_, b_, seed * 100003 + j)
            end = time.perf_counter()
            st['lat'].append(end - arrive)
            policy.get('done', lambda st: None)(st)
            recs_.append(dict(j=j, q=q, a=a, lam=lam, arrive=arrive - t0, end=end - t0, latency=end - arrive,
                              decide_ms=(decided - arrive) * 1000, ok=out['ok'], tokens=out['tokens'], error=out['error'],
                              correct=bool(out['ok'] and vote_correct(out['preds'], GOLD[q], st['vote_rng'])),
                              waiting=snap.get('waiting'), running=snap.get('running')))

        tasks = []
        for j, (q, ta) in enumerate(zip(qs, ts)):
            await asyncio.sleep(max(0.0, t0 + ta - time.perf_counter()))
            tasks.append(asyncio.create_task(one(j, q)))
        await asyncio.gather(*tasks)
        stop.set()
        await mon
    for r in tel:
        r['t'] -= t0
    res = dict(tag=tag, seed=seed, rep=rep, spec=list(spec), records=sorted(recs_, key=lambda r: r['j']), telemetry=tel)
    json.dump(res, open(path, 'w'))
    return res


def window(m):
    return max(r['arrive'] for r in m) - min(r['arrive'] for r in m)


def summary(res, warmup=None, slo=SLO):
    m = res['records'][len(res['records']) // 10 if warmup is None else warmup:]
    lat = np.array([r['latency'] for r in m])
    win, t_half, t_last = window(m), np.median([r['arrive'] for r in m]), m[-1]['arrive']
    tel = [t for t in res['telemetry'] if t.get('waiting') is not None]
    half = [t for t in tel if t_half <= t['t'] <= t_last] or tel
    return dict(goodput=sum(r['correct'] and r['latency'] <= slo for r in m) / win,
                accuracy=np.mean([r['correct'] for r in m]), attain=np.mean(lat <= slo),
                p95=np.percentile(lat, 95), tokens=np.mean([r['tokens'] for r in m]),
                errors=sum(not r['ok'] for r in m), decide_ms=np.mean([r['decide_ms'] for r in m]),
                throughput_ratio=sum(t_half <= r['end'] <= t_last for r in m) / max(sum(r['arrive'] >= t_half for r in m), 1),
                waiting=np.mean([t['waiting'] for t in half]) if half else np.nan)


def fixed(a):
    async def decide(q, st, now):
        return a, None
    return dict(decide=decide)


In [ ]:
# Step A: background-only rate ladder with the reference action -> light and contended rates
L = CFG['ladder']
REF = ACTIONS.index(tuple(L['reference_action']))
cal = WD / 'burst.json'
if not cal.exists():
    async with session() as s:
        t0 = time.perf_counter()
        await asyncio.gather(*(serve(s, TEXT[q], *ACTIONS[REF], 9 + i) for i, q in enumerate(DEV[:L['burst']])))
        json.dump({'rps': L['burst'] / (time.perf_counter() - t0)}, open(cal, 'w'))
BURST = json.load(open(cal))['rps']

ladder = []
for mult in L['multipliers']:
    rate = mult * BURST
    res = await run(f'A_ladder_{mult}', fixed(REF), ('const', rate), 0, DEV, max(20, int(rate * L['window_s'])))
    ladder.append(dict(mult=mult, rate=rate, **summary(res)))
ladder = pd.DataFrame(ladder)
print(ladder[['mult', 'rate', 'waiting', 'throughput_ratio', 'p95', 'attain', 'goodput']].round(3).to_string(index=False))
quiet = ladder[ladder.waiting == 0]
queued = ladder[(ladder.waiting >= 1) & (ladder.throughput_ratio >= 0.95)]
assert len(quiet) and len(queued), 'no light and/or stable contended setting on this ladder: report this, or extend the ladder'
RATE = {'light': float(quiet.rate.max()), 'contended': float(queued.rate.min())}
print('frozen rates (req/s):', RATE)


In [ ]:
# Step B: every fixed action at light and contended load on dev questions -> best fixed per load, live vs label accuracy
B = []
for load, rate in RATE.items():
    for a in range(NA):
        res = await run(f'B_{load}_a{a}', fixed(a), ('const', rate), 0, DEV, CFG['step_b']['n_requests'])
        B.append(dict(load=load, rate=rate, a=a, action=aname(a), **summary(res)))
B = pd.DataFrame(B)
B['label_acc'] = [PF[DEV, a].mean() for a in B.a]
print(B[['load', 'action', 'goodput', 'accuracy', 'label_acc', 'attain', 'p95', 'tokens', 'waiting']].round(3).to_string(index=False))
BEST_FIXED = {load: int(g.loc[g.goodput.idxmax(), 'a']) for load, g in B.groupby('load')}
TOK = B.groupby('a').tokens.mean().reindex(range(NA)).to_numpy()
print('best fixed action per load:', {k: aname(a) for k, a in BEST_FIXED.items()})


In [ ]:
# Step C: p95 model = f(arrival rate x tokens per request); leave-one-action-out check near the target
cells = [(r.rate * r.tokens, r.p95, r.a) for r in B.itertuples()] + [(r.rate * TOK[REF], r.p95, REF) for r in ladder.itertuples()]


def curve(pts):
    pts = sorted(pts)
    x = np.array([p[0] for p in pts]); y = np.maximum.accumulate([p[1] for p in pts])

    def f(v):
        v = np.asarray(v, float)
        slope = (y[-1] - y[-2]) / max(x[-1] - x[-2], 1e-9) if len(x) > 1 else 0.0
        return np.where(v > x[-1], y[-1] + slope * (v - x[-1]), np.interp(v, x, y))
    return f


C = CFG['step_c']
rows = []
for a in range(NA):
    f = curve([c for c in cells if c[2] != a])
    rows += [dict(action=aname(a), measured=y, predicted=float(f(x)))
             for x, y, aa in cells if aa == a]
chk = pd.DataFrame(rows)
chk['error'] = chk.predicted - chk.measured
near = chk[(chk.measured - SLO).abs() <= C['near_band'] * SLO]
PASS_C = bool(len(near)) and bool((near.error.abs() <= C['tolerance'] * SLO).all())
MARGIN = 0.0 if PASS_C else float(max(0.0, -near.error.min())) if len(near) else float(max(0.0, -chk.error.min()))
P95 = curve(cells)
p95_model = lambda x: P95(x) + MARGIN
print(chk.round(2).to_string(index=False))
print(f'near-target cells: {len(near)} | Step C {"PASS" if PASS_C else "FAIL"} | safety margin added: {MARGIN:.1f} s')


In [ ]:
# Step D0: policies tuned on dev only (best fixed, fixed mixture, frozen-cost router, load-aware router)
LAMS = np.concatenate([[0.0], np.logspace(-3, 2, 60)])
TOKN = TOK / TOK.mean()
TARGET = SLO * CFG['slo_safety']


def best_lambda(rate):
    best = (-1, LAMS[-1])
    for lam in LAMS:
        ch = np.argmax(PHAT[DEV] - lam * TOKN, axis=1)
        if p95_model(rate * TOK[ch].mean()) <= TARGET:
            best = max(best, (PF[DEV, ch].mean(), lam))
    return best[1]


def best_mixture(rate):
    acc = PF[DEV].mean(0)
    best = (-1, 0, 0, 1.0)
    for a1 in range(NA):
        for a2 in range(NA):
            for p in np.linspace(0, 1, 11):
                if p95_model(rate * (p * TOK[a1] + (1 - p) * TOK[a2])) <= TARGET:
                    best = max(best, (p * acc[a1] + (1 - p) * acc[a2], a1, a2, p))
    return best[1:]


RGRID = np.linspace(0.5 * RATE['light'], 1.5 * RATE['contended'], 40)
LAM_GRID = np.array([best_lambda(r) for r in RGRID])
LAM_FROZEN = best_lambda(RATE['light'])
MIX = {load: best_mixture(r) for load, r in RATE.items()}


def mixture(a1, a2, p):
    async def decide(q, st, now):
        return (a1 if st['rng'].random() < p else a2), None
    return dict(decide=decide)


def router(adaptive):
    R = CFG['router']

    async def decide(q, st, now):
        if adaptive:
            while st['arrivals'] and now - st['arrivals'][0] > R['rate_window_s']:
                st['arrivals'].popleft()
            span = min(R['rate_window_s'], max(now - st['t0'], 1.0))
            lam = float(np.interp(len(st['arrivals']) / span, RGRID, LAM_GRID)) * st['mult']
        else:
            lam = LAM_FROZEN
        ph = await probe(q)
        return int(np.argmax(ph - lam * TOKN)), lam

    def done(st):
        if adaptive and len(st['lat']) == st['lat'].maxlen:
            p = np.percentile(st['lat'], 95)
            st['mult'] = float(np.clip(st['mult'] * (R['feedback_step'] if p > TARGET else
                                       1 / R['feedback_step'] if p < 0.6 * SLO else 1), 0.25, 4))
    return dict(decide=decide, done=done)


def policies(cond):
    load = 'contended' if cond == 'ramp' else cond
    return {'best_fixed': fixed(BEST_FIXED[load]), 'fixed_mixture': mixture(*MIX[load]),
            'frozen_router': router(False), 'load_aware': router(True)}


def spec(cond):
    return ('ramp', RATE['light'], RATE['contended']) if cond == 'ramp' else ('const', RATE[cond])


print('frozen lambda:', LAM_FROZEN, '| mixtures (a1, a2, P(a1)):', {k: (aname(v[0]), aname(v[1]), round(float(v[2]), 1)) for k, v in MIX.items()})


In [ ]:
# Step D1: A/A pairs (same seed within a pair, new seed per pair) -> noise, MDE, minimum meaningful gain (frozen)
E, M = CFG['e2'], CFG['mmg']
mmg_path = WD / 'mmg.json'
if mmg_path.exists():
    MMG = json.load(open(mmg_path))
else:
    MMG = {}
    for load in ('light', 'contended'):
        d, g = [], []
        for k in range(CFG['aa']['n_pairs']):
            sd = CFG['aa']['seed_start'] + k
            pair = []
            for rep in (0, 1):
                res = await run(f'AA_{load}', fixed(BEST_FIXED[load]), spec(load), sd, HELD, E['n_requests'], rep)
                pair.append(summary(res, E['warmup'])['goodput'])
            d.append(pair[0] - pair[1]); g += pair
        mde = M['z_total'] * np.std(d, ddof=1) / math.sqrt(len(E['seeds']))
        MMG[load] = dict(aa_diffs=d, sd=float(np.std(d, ddof=1)), mde=float(mde), baseline_goodput=float(np.mean(g)),
                         mmg=float(max(mde, M['practical_floor'] * np.mean(g))))
    MMG['ramp'] = MMG['contended']
    json.dump(MMG, open(mmg_path, 'w'), indent=1)
print(pd.DataFrame(MMG).T[['sd', 'mde', 'baseline_goodput', 'mmg']].round(4))


In [ ]:
# Step D2: E2 runs on held-out questions (resumable; finished runs are skipped)
for seed in E['seeds']:
    for cond in E['conditions']:
        for name, pol in policies(cond).items():
            res = await run(f'E2_{cond}_{name}', pol, spec(cond), seed, HELD, E['n_requests'])
            s_ = summary(res, E['warmup'])
            print(f"s{seed} {cond:9s} {name:14s} goodput={s_['goodput']:.3f} acc={s_['accuracy']:.3f} "
                  f"attain={s_['attain']:.2f} p95={s_['p95']:.1f}s waiting={s_['waiting']:.1f}", flush=True)


In [ ]:
# Step D3: results and pre-registered verdict (load-aware must beat every baseline by >= MMG, CI above 0, at contended and ramp)
def load_run(cond, name, seed):
    return json.load(open(RUNS / f'E2_{cond}_{name}__s{seed}__r0.json'))


NAMES = list(policies('light'))
tab = pd.DataFrame([dict(cond=c, policy=n, seed=s, **summary(load_run(c, n, s), E['warmup']))
                    for c in E['conditions'] for n in NAMES for s in E['seeds']])
print(tab.groupby(['cond', 'policy'])[['goodput', 'accuracy', 'attain', 'p95', 'tokens', 'decide_ms', 'waiting']]
      .mean().round(3).to_string())

boot = np.random.default_rng(0)
cmp_rows = []
for c in E['conditions']:
    for base_ in NAMES[:-1]:
        hits = []
        for s in E['seeds']:
            x, y = (load_run(c, n_, s)['records'][E['warmup']:] for n_ in ('load_aware', base_))
            cw = lambda m: np.array([r['correct'] and r['latency'] <= SLO for r in m], float) / window(m)
            hits.append(cw(x) - cw(y))
        point = np.mean([h.sum() for h in hits])
        draws = [np.mean([h[boot.integers(0, len(h), len(h))].sum() for h in hits]) for _ in range(2000)]
        lo, hi = np.percentile(draws, [2.5, 97.5])
        cmp_rows.append(dict(cond=c, vs=base_, gain=point, ci_low=lo, ci_high=hi, mmg=MMG[c]['mmg'],
                             passed=bool(point >= MMG[c]['mmg'] and lo > 0)))
cmp_df = pd.DataFrame(cmp_rows)
print(cmp_df.round(4).to_string(index=False))
decisive = cmp_df[cmp_df.cond.isin(['contended', 'ramp'])]
att = tab.groupby(['cond', 'policy']).attain.mean()
if decisive.passed.all():
    verdict = 'LATTE WORKS: load-aware beats every baseline by at least the MMG at contended load and on the ramp.'
elif all(att[c, 'load_aware'] > max(att[c, b] for b in NAMES[:-1]) for c in ('contended', 'ramp')):
    verdict = 'SAFETY ONLY: no meaningful goodput gain, but load-aware meets the latency target more often.'
else:
    verdict = 'LATTE NOT SUPPORTED: no meaningful gain over at least one baseline at contended load or on the ramp.'
print(verdict)
tab.to_csv(WD / 'e2_runs.csv', index=False); cmp_df.to_csv(WD / 'e2_comparisons.csv', index=False)
(WD / 'verdict.txt').write_text(verdict + '\n')
